STEP 01: IMPORTING LIBRARIES

In [13]:
import os
from dotenv import load_dotenv

import pandas as pd
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import chroma
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

True

In [14]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY")



In [15]:
model = init_chat_model("groq:qwen/qwen3.6-27b")
response = model.invoke("what is anime?")
response.content

APIConnectionError: Connection error.

STEP 02: DATA INGESTION PIPELINE

In [9]:
#DATA LOADING
class AnimeDataLoader:
    def __init__(self, original_csv: str, processed_csv: str):
        self.original_csv = original_csv
        self.processed_csv = processed_csv
    
    def load_and_process(self):
        df = pd.read_csv(
            self.original_csv,
            encoding="utf-8",
            on_bad_lines="skip"
        ).dropna()
        
        required_cols = {"Name" , "Genres" , "Sypnopsis"}
        if not required_cols.issubset(df.columns):
            raise ValueError("Missing required columns in CSV")
        
        df["combined_info"] = (
            "Title: " + df["Name"]
            + "Overview: " + df["sypnopsis"]
            + "Genres" + df["Genres"]
        )
        
        df[["combined_info"]].to_csv(
            self.processed_csv,
            index=False,
            encoding="utf-8",
        )
        
        return self.processed_csv
                

In [ ]:
#SPLIT AND STORE THE DATA

class VectorStoreBuilder:
    def __init__(self, csv_path: str, persist_dir:str = "chroma_db"):
        self.csv_path = csv_path,
        self.persist_dir = persist_dir,
        self.embedding = HuggingFaceEmbeddings(
            model_name = "all-MiniLM"
        )
        